# Causal Forest multi-output — fit and save

This notebook trains per-endpoint causal forest models (an alternative to the T-learner in
`04_fit_calibrated_models.ipynb`). It fits **three explicit flexibility levels** of the base
model — `regularized` (shallow, heavily regularized), `medium`, and `flexible` (deep, high
capacity) — defined in `casual_multioutput_pipeline.CF_MODEL_PRESETS`. Each preset scales both
the causal-forest hyper-parameters and the first-stage LGBM nuisances. For each level it:

1. trains a multi-output `CausalMultiOutputPipeline` on the full data;
2. **saves** it to `models/CausalForest/CausalForest_multioutput_<level>.joblib` (the `medium`
   model is also saved as `CausalForest_multioutput.joblib`, the default 07 loads).

The three levels double as a sensitivity analysis: if no CATE heterogeneity survives even the
`flexible` model — while `regularized` collapses toward the ATE — the homogeneous-effect
conclusion is robust to model capacity. Optuna tuning is retained behind `RUN_OPTUNA` for
reference but no longer drives the saved models.

The first-stage (nuisance) learners `model_y=E[Y|x]` and `model_t=E[T|x]` travel with each saved
object, so the analysis notebook reuses the exact same configuration instead of re-hardcoding it.

The CATE estimation, its visualisations and the predictive evaluation (out-of-fold ROC,
held-out confusion matrices, heterogeneity tests) live in `07_causal_forest_analysis.ipynb`.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna

from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_curve, roc_auc_score
from econml.dml import CausalForestDML
from lightgbm import LGBMRegressor, LGBMClassifier
from IPython.display import display

from thesis_utils import predict_proba_matrix, plot_confusion_matrices

DATA_DIR = os.path.normpath(os.path.join(os.getcwd(), '..', '..', 'data'))
MODELS_DIR = os.path.normpath(os.path.join(os.getcwd(), '..', 'models'))

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

/home/tpioda/Bachelor-Thesis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

`USE_KNN_IMPUTER` controls imputation strategy. The saved models come from the three explicit
`CF_MODEL_PRESETS`; `DEFAULT_LEVEL` is the one mirrored to the canonical filename. Optuna tuning
is optional and gated by `RUN_OPTUNA` (`TUNE_CV` folds, `N_TRIALS` trials, overridable via
`CATE_N_TRIALS`).

In [2]:
USE_KNN_IMPUTER = True
KNN_NEIGHBORS = 5

RUN_OPTUNA = False                  # the three explicit presets drive the saved models;
                                    # flip on to explore tuned hyper-parameters for reference
TUNE_CV = 3                         # CV folds for the Optuna objective
N_TRIALS = int(os.environ.get('CATE_N_TRIALS', 5))   # Optuna trials

DEFAULT_LEVEL = 'medium'            # preset also saved as the canonical default model

CALIBRATION_METHOD = 'sigmoid'      # Only for reference; CausalForest does not use it

print(f'Imputation strategy        : {"KNN" if USE_KNN_IMPUTER else "Median"}')
print(f'Optuna tuning              : {"on" if RUN_OPTUNA else "off (using presets)"}')
print(f'Optuna trials              : {N_TRIALS}')
print(f'CV folds for tuning        : {TUNE_CV}')

Imputation strategy        : KNN
Optuna tuning              : off (using presets)
Optuna trials              : 5
CV folds for tuning        : 3


## Load data and split by treatment arm

Load the cleaned features and 5 endpoints, and encode treatment as 0 (abbreviated DAPT) /
1 (prolonged DAPT).

In [3]:
X = pd.read_parquet(os.path.join(DATA_DIR, 'X_features.parquet'))
y_df = pd.read_parquet(os.path.join(DATA_DIR, 'y_targets.parquet'))

TARGETS = [
    'cec_barc235_335d',
    'cec_cvdeath_335d',
    'cec_mi_335d',
    'cec_stroke_335d',
    'cec_bleed_335d',
]
TARGET_LABELS = ['barc_235', 'death', 'mi', 'stroke', 'bleed']
y_df = y_df[TARGETS]

X_num = X.select_dtypes(include=[np.number])

regimen = X['regimen']
T = regimen.map({'prolonged DAPT': 1, 'abbreviated DAPT': 0}).to_numpy()

print(f'Samples: {len(X_num)}, Features: {X_num.shape[1]}')
print(f'Treatment: {np.sum(T)} prolonged DAPT, {len(T) - np.sum(T)} abbreviated DAPT')
print(f'Targets: {len(TARGETS)} endpoints')
print(f'Event rates: {dict(y_df.mean().round(3))}')

Samples: 4579, Features: 63
Treatment: 2284 prolonged DAPT, 2295 abbreviated DAPT
Targets: 5 endpoints
Event rates: {'cec_barc235_335d': np.float64(0.078), 'cec_cvdeath_335d': np.float64(0.018), 'cec_mi_335d': np.float64(0.024), 'cec_stroke_335d': np.float64(0.008), 'cec_bleed_335d': np.float64(0.111)}


## Custom Causal Multi-output Pipeline

Wrapper around per-endpoint CausalForestDML with robust imputation and preprocessing.
Trains one causal forest per target; `predict_cate` returns shape `(n_samples, n_targets)`.

In [4]:
from casual_multioutput_pipeline import (
    CausalMultiOutputPipeline,
    CF_MODEL_PRESETS,
    make_preset_pipeline,
)

print('Base-model flexibility presets:')
for level, preset in CF_MODEL_PRESETS.items():
    print(f"  {level:12s} forest={preset['cf_params']}")

Base-model flexibility presets:
  regularized  forest={'n_estimators': 400, 'max_depth': 2, 'min_samples_leaf': 40, 'max_samples': 0.3, 'min_balancedness_tol': 0.3, 'min_impurity_decrease': 0.001}
  medium       forest={'n_estimators': 500, 'max_depth': 4, 'min_samples_leaf': 15, 'max_samples': 0.4}
  flexible     forest={'n_estimators': 600, 'max_depth': 8, 'min_samples_leaf': 5, 'max_samples': 0.5}


## Hyper-parameter tuning with Optuna (optional, `RUN_OPTUNA`)

Reference only — the saved models come from the three presets, not from tuning. When enabled,
the TPE sampler searches the causal forest hyper-parameters, minimising the mean **negative
R-score** (an R²-analogue for treatment effects, Nie & Wager 2020) across endpoints, estimated
via `TUNE_CV`-fold CV — i.e. rewarding forests whose CATEs best explain the residualised
outcome. The first-stage (nuisance) learners are held fixed at the pipeline's
`make_model_y`/`make_model_t` configuration.

In [5]:
import random
def make_objective(X, Y, T):
    def objective(trial):
        cf_params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 600, step=4*25), #based on the 'sub_forest_size' parameter default 4
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 20),
            'max_samples': trial.suggest_float('max_samples', 0.3, 0.5),
            'inference': False,  # Set to True if you want confidence intervals
        }

        while True:
            sub_forest_size = random.randint(3,9)
            if cf_params['n_estimators'] % sub_forest_size == 0:
                cf_params['subforest_size'] = sub_forest_size
                break

        pipeline = CausalMultiOutputPipeline(
            use_knn_imputer=USE_KNN_IMPUTER,
            knn_neighbors=KNN_NEIGHBORS,
            cf_params=cf_params
        )
        return pipeline.score_cv(X, Y, T, cv=TUNE_CV)
    return objective

if RUN_OPTUNA:
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(X_num.values, y_df.values, T), n_trials=N_TRIALS, show_progress_bar=True)
    print(f'Best CV score: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')
else:
    print('Optuna tuning skipped (RUN_OPTUNA=False) — using the three explicit presets below.')

Optuna tuning skipped (RUN_OPTUNA=False) — using the three explicit presets below.


## Fit the three flexibility levels and save

Train one `regularized`, one `medium`, and one `flexible` causal forest on the full dataset with
inference (bootstrap CIs) enabled, and save each to its own file. The `DEFAULT_LEVEL` model is
also mirrored to `CausalForest_multioutput.joblib` so `07` keeps loading a canonical default.

In [6]:
SAVE_DIR = os.path.join(MODELS_DIR, 'CausalForest')
os.makedirs(SAVE_DIR, exist_ok=True)

# Re-saving all levels: drop any stale causal-forest files first.
for stale in glob.glob(os.path.join(SAVE_DIR, 'CausalForest_*.joblib')):
    os.remove(stale)

pipelines = {}
for level in CF_MODEL_PRESETS:
    print(f'Fitting "{level}" causal forest ...')
    pipe = make_preset_pipeline(
        level, inference=True,
        use_knn_imputer=USE_KNN_IMPUTER, knn_neighbors=KNN_NEIGHBORS,
    )
    pipe.fit(X_num.values, y_df.values, T)
    pipelines[level] = pipe
    joblib.dump(pipe, os.path.join(SAVE_DIR, f'CausalForest_multioutput_{level}.joblib'))
    print(f'  saved -> models/CausalForest/CausalForest_multioutput_{level}.joblib')

# Mirror the default level to the canonical filename 07 loads.
joblib.dump(pipelines[DEFAULT_LEVEL], os.path.join(SAVE_DIR, 'CausalForest_multioutput.joblib'))
print(f'\nDefault ({DEFAULT_LEVEL}) mirrored -> models/CausalForest/CausalForest_multioutput.joblib')
print('Done. Run 07_causal_forest_analysis.ipynb to analyse the effects.')

Fitting "regularized" causal forest ...
  saved -> models/CausalForest/CausalForest_multioutput_regularized.joblib
Fitting "medium" causal forest ...
  saved -> models/CausalForest/CausalForest_multioutput_medium.joblib
Fitting "flexible" causal forest ...
  saved -> models/CausalForest/CausalForest_multioutput_flexible.joblib

Default (medium) mirrored -> models/CausalForest/CausalForest_multioutput.joblib
Done. Run 07_causal_forest_analysis.ipynb to analyse the effects.


## Estimate CATE on full cohort — compare flexibility levels

Generate CATE predictions for all patients under each preset. The key diagnostic is `std_CATE`:
it should grow from `regularized` to `flexible`. If even the `flexible` model keeps it small,
the lack of heterogeneity is robust to model capacity rather than a regularization artefact.

In [7]:
summaries = {}
for level, pipe in pipelines.items():
    cate_preds = pipe.predict_cate(X_num.values)
    cate = pd.DataFrame(cate_preds, columns=TARGET_LABELS, index=X_num.index)
    summaries[level] = pd.DataFrame({
        'mean_CATE': cate.mean(),
        'std_CATE': cate.std(),
        'pct_lDAPT_better': (cate < 0).mean() * 100,
        'pct_sDAPT_better': (cate > 0).mean() * 100,
    })
    print(f'\nCATE summary — {level}')
    display(summaries[level].round(4))

# Side-by-side CATE dispersion across levels (heterogeneity signal).
std_by_level = pd.DataFrame({lvl: s['std_CATE'] for lvl, s in summaries.items()})
print('\nstd(CATE) by flexibility level')
display(std_by_level.round(4))


CATE summary — regularized


,mean_CATE,std_CATE,pct_lDAPT_better,pct_sDAPT_better
barc_235,0.028,0.002,0.000,100.000
death,0.003,0.000,0.000,100.000
mi,-0.004,0.000,100.000,0.000
stroke,0.004,0.000,0.000,100.000
bleed,0.046,0.004,0.000,100.000



CATE summary — medium


,mean_CATE,std_CATE,pct_lDAPT_better,pct_sDAPT_better
barc_235,0.026,0.005,0.000,100.000
death,0.003,0.004,13.933,86.067
mi,-0.005,0.004,93.754,6.246
stroke,0.004,0.003,2.468,97.532
bleed,0.045,0.006,0.000,100.000



CATE summary — flexible


,mean_CATE,std_CATE,pct_lDAPT_better,pct_sDAPT_better
barc_235,0.026,0.014,2.883,97.117
death,0.004,0.011,27.801,72.199
mi,-0.005,0.012,76.392,23.608
stroke,0.004,0.006,12.208,87.792
bleed,0.045,0.016,0.764,99.236



std(CATE) by flexibility level


,regularized,medium,flexible
barc_235,0.002,0.005,0.014
death,0.000,0.004,0.011
mi,0.000,0.004,0.012
stroke,0.000,0.003,0.006
bleed,0.004,0.006,0.016
